In [1]:
import xarray as xr
import numpy as np

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
from glob import glob
from scipy import stats 
import matplotlib.gridspec as gridspec

In [2]:
from utils.regridding import Regridder, create_target_grid
from utils.stats import jja, area_weighted_ave, compute_trend_ols 

In [3]:
# ======================================================================
# Change these
# ======================================================================
# The directory where the main git repo is located (the one that contains the "data" and "scripts" folders)
WDIR = "/project/tas1/itbaxter/for-tiffany/IPCC_figures/lesfmip_figures"

# The grid to regrid to
target_lat = np.arange(-90, 90.1, 1)
target_lon = np.arange(0, 360, 1)

In [ ]:
# Change if you want to use a different file than the one downloaded with get_era5_plevs.py
era5_file = f"{WDIR}/raw_data/Reanalysis/era5_monthly_u_component_of_wind_1979-2023.nc"

era5_uwind = xr.open_dataset(era5_file).rename({"valid_time": "time",
                                                "longitude": "lon",
                                                "latitude": "lat",
                                                "pressure_level": "plev",
                                                }).drop("number").drop("expver").sortby("lat").sel(plev=200, method='nearest')
print(era5_uwind)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 540, lat: 721, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2023-12-01
    plev     float64 8B 200.0
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon      (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
Data variables:
    u        (time, lat, lon) float32 2GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-12T15:38 GRIB to CDM+CF via cfgrib-0.9.1...


/home/itbaxter/ipykernel_3725768/2491975072.py:8: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  }).drop("number").drop("expver").sortby("lat").sel(plev=200, method='nearest')


: 

In [ ]:
# Regrid to 1 degree
target_grid = create_target_grid(None, 
                                 target_lat=target_lat, 
                                 target_lon=target_lon)

era5_uwind_1deg = Regridder(era5_uwind, target_grid, method='conservative').regrid()
print(era5_uwind_1deg)

Regridding...


/project/tas1/itbaxter/miniforge3/envs/climadyn/lib/python3.13/site-packages/xesmf/backend.py:56: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')
/project/tas1/itbaxter/miniforge3/envs/climadyn/lib/python3.13/site-packages/xesmf/backend.py:56: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')


In [ ]:
# N96 grid
target_lat = np.arange(-90, 90.1, 1.25)
target_lon = np.arange(0, 360, 1.875)

target_grid = create_target_grid(None, 
                                 target_lat=target_lat, 
                                 target_lon=target_lon)

era5_uwind_n96 = Regridder(era5_uwind, target_grid, method='conservative').regrid()
print(era5_uwind_n96)

In [ ]:
# 2.5 degree grid 
target_lat = np.arange(-90, 90.1, 2.5)
target_lon = np.arange(0, 360, 2.5)

target_grid = create_target_grid(None, 
                                 target_lat=target_lat, 
                                 target_lon=target_lon)

era5_uwind_2_5deg = Regridder(era5_uwind, target_grid, method='conservative').regrid()
print(era5_uwind_2_5deg)

In [ ]:
era5_eswj_1deg = area_weighted_ave(jja(era5_uwind_1deg.sel(lat=slice(35,45), lon=slice(30, 120))))
era5_eswj_n96 = area_weighted_ave(jja(era5_uwind_n96.sel(lat=slice(35,45), lon=slice(30, 120))))
era5_eswj_2_5deg = area_weighted_ave(jja(era5_uwind_2_5deg.sel(lat=slice(35,45), lon=slice(30, 120))))

In [ ]:
era5_eswj_1deg_trend = compute_trend_ols(era5_eswj_1deg['u'].sel(year=slice(1979, 2019)), dim='year')
era5_eswj_n96_trend = compute_trend_ols(era5_eswj_n96['u'].sel(year=slice(1979, 2019)), dim='year')
era5_eswj_2_5deg_trend = compute_trend_ols(era5_eswj_2_5deg['u'].sel(year=slice(1979, 2019)), dim='year')

In [ ]:
def rmac(da, year0=1979, year1=2019):
    return da - da.sel(year=slice(year0, year1)).mean('year')

In [ ]:
fig = plt.figure(figsize=(10, 5))

gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], width_ratios=[3, 1], hspace=0.3, wspace=0.2, left=0.08, right=0.90, top=0.96, bottom=0.10)

ax = fig.add_subplot(gs[0, 0])

ax.plot(era5_eswj_1deg['year'], rmac(era5_eswj_1deg['u']), c='k', label=' (1deg)')
ax.plot(era5_eswj_n96['year'], rmac(era5_eswj_n96['u']), c='r', marker='o', ls='--', lw=0.8, label='ERA5 (N96)')
ax.plot(era5_eswj_2_5deg['year'], rmac(era5_eswj_2_5deg['u']), c='b', marker='s', ls=' ', label='ERA5 (2.5deg)')

years = np.arange(1979, 2020)
ax.plot(years, rmac(era5_eswj_1deg_trend['predicted']), c='k', ls='--', lw=0.8, label=f"Trend (1deg)")
ax.plot(years, rmac(era5_eswj_n96_trend['predicted']), c='r', ls='--', lw=0.8, label=f"Trend (N96)")
ax.plot(years, rmac(era5_eswj_2_5deg_trend['predicted']), c='b', ls='--', lw=0.8, label=f"Trend (2.5deg)")

ax.set_xlabel('Year')
ax.set_xlim([1979, 2023])
ax.set_ylim([-6,6])
ax.minorticks_on()
ax.legend(frameon=False, ncols=2, fontsize=8)
ax.axhline(0, c='silver', ls='--', lw=0.6)
ax.set_title('Zonal wind over(35N-45N, 30E-120E) at 200 hPa')
ax.text(-0.07, 1.0, "a", fontsize=16, transform=ax.transAxes)

ax = fig.add_subplot(gs[0, 1])
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(['Obs', 'ALL', 'GHG', 'AER', 'NAT'])
ax.set_ylim([-1, 1])
ax.set_xlim([-0.5, 4.5])
ax.set_ylabel('Linear trend')
ax.axhline(0, c='silver', ls='--', lw=0.4)
ax.text(-0.28, 1.0, "b", fontsize=16, transform=ax.transAxes)
ax.minorticks_on()
ax.xaxis.set_tick_params(which='minor', bottom=False)

ax.scatter([0.1], [10*era5_eswj_1deg_trend['slope'].values], marker='d', facecolor='none', edgecolor='gray')
ax.scatter([0], [10*era5_eswj_n96_trend['slope'].values], marker='d', facecolor='none', edgecolor='gray',)
ax.scatter([-0.1], [10*era5_eswj_2_5deg_trend['slope'].values], marker='d', facecolor='none', edgecolor='gray')

plt.savefig(f"{WDIR}/plots/Dong_et_al_2024-ESWJ_timseries_and_trends.png", dpi=300)
plt.savefig(f"{WDIR}/plots/Dong_et_al_2024-ESWJ_timseries_and_trends.pdf", dpi=300)